# Proyecto Detección de Bloqueos en Calles
## Filtrado de datos

### BLOQUEOS
- **Bloqueos**: Imagenes de bloqueos recolectadas usando icrawler, al rededor de 300 imagenes de personas bloqueando, manifestantes recolectadas de internet.  
Dataset de Roboflow
- **Conos**: Imagenes de conos en calles  
- **Llantas**: Imagenes de llantas en calles como parte de posibles cosas que pueden usarse para bloquear  
- **Accidentes**: Imagenes de accidentes de autos que pueden identificarse como parte de un bloqueo  

### NO BLOQUEOS
- **Trafico**: Imagenes de un trafico normal para destinguir cuando es un bloqueo
- **BDD100K**: Imagenes captadas para lo que es conduccion autonoma que cuenta con trafico y flujo normal

In [1]:
import os
import shutil
import random
from pathlib import Path
from torchvision import transforms
from PIL import Image

In [9]:
BASE = Path.cwd().parents[0] / "dataset" / "imagenes_dataset"
carpetas = ["01_CONOS", "02_TRAFFIC_FLOW", "03_LLANTAS", "04_ACCIDENTES", "05_BDD100K", "BLOQUEOS"]

for c in carpetas:
    total = 0
    for split in ["train", "valid", "test"]:
        p = BASE / c / split / "images"
        if p.exists():
            n = len(list(p.glob("*")))
            total += n
    print(f"{c}: {total} imágenes")

01_CONOS: 1867 imágenes
02_TRAFFIC_FLOW: 726 imágenes
03_LLANTAS: 1636 imágenes
04_ACCIDENTES: 9891 imágenes
05_BDD100K: 1484 imágenes
BLOQUEOS: 0 imágenes


Division de las imagenes de Bloqueos por personas obtenidas por icrawler, se dividio y se agrego a la carpeta dataset diviendola en un 70 % train, 15% test, 15% val 

In [10]:
random.seed(42)

BASE = Path.cwd().parents[0] / "dataset" / "imagenes_dataset"
DESTINO = Path.cwd().parents[0] / "dataset" / "dataset_bloqueos"

EXT_VALIDAS = [".jpg", ".jpeg", ".png"]
splits = {"train": 0.7, "val": 0.15, "test": 0.15}  # --> Division para train 70%, val 15% y test 15%

def recolectar_imagenes_plano(carpeta_dataset, max_samples=None):
    p = BASE / carpeta_dataset  
    imgs = [f for f in p.glob("*") if f.suffix.lower() in EXT_VALIDAS]     
    if max_samples is not None and len(imgs) > max_samples:  
        imgs = random.sample(imgs, max_samples) 
    return imgs

imgs_bloqueos_originales = recolectar_imagenes_plano("BLOQUEOS")
print(f"BLOQUEOS originales encontradas: {len(imgs_bloqueos_originales)}")
random.shuffle(imgs_bloqueos_originales)
n = len(imgs_bloqueos_originales)
n_train = int(n * splits["train"])
n_val = int(n * splits["val"])

split_bloqueos = {
    "train": imgs_bloqueos_originales[:n_train],
    "val": imgs_bloqueos_originales[n_train:n_train + n_val],
    "test": imgs_bloqueos_originales[n_train + n_val:],
}

print(f"train={len(split_bloqueos['train'])}, val={len(split_bloqueos['val'])}, test={len(split_bloqueos['test'])}")
nombres_por_split = {"train": set(), "val": set(), "test": set()}

for split, lista in split_bloqueos.items():
    carpeta_out = DESTINO / split / "bloqueo"
    carpeta_out.mkdir(parents=True, exist_ok=True)
    for img_path in lista:
        shutil.copy(img_path, carpeta_out / img_path.name)
        nombres_por_split[split].add(img_path.stem) 


BLOQUEOS originales encontradas: 303
train=212, val=45, test=46


Generacion de mas datos rotandolos y demás, dado que el dataset de gente bloqueando solo contaba con 300 imagenes, esta rotacion se aplica solo al de entrenamiento

In [11]:
DESTINO = Path.cwd().parents[0] / "dataset" / "dataset_bloqueos"
carpeta_train = DESTINO / "train" / "bloqueo"

transformaciones = [
    transforms.Compose([
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
    ]),
    transforms.Compose([
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3, contrast=0.2, saturation=0.2),
    ]),
    transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
    ]),
    transforms.Compose([
        transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),
        transforms.ColorJitter(brightness=0.25, contrast=0.25),
    ]),
]

originales = [f for f in carpeta_train.glob("*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
print(f"Imágenes originales en train/bloqueo: {len(originales)}")

n_generadas = 0
for img_path in originales:
    img = Image.open(img_path).convert("RGB")
    for i, t in enumerate(transformaciones):
        img_aug = t(img)
        img_aug = img_aug.resize((224, 224))
        nuevo_nombre = f"{img_path.stem}_aug{i}{img_path.suffix}"
        img_aug.save(carpeta_train / nuevo_nombre)
        n_generadas += 1

print(f"Generadas {n_generadas} imágenes nuevas")
print(f"Total en train/bloqueo ahora: {len(list(carpeta_train.glob('*')))}")

Imágenes originales en train/bloqueo: 212
Generadas 848 imágenes nuevas
Total en train/bloqueo ahora: 1060


Todo el conjunto de imagenes se los dividio en la carpeta de dataset llevando de las imagenes de conos, llantas, accidentes a las carpetas de bloqueos y de las de trafico y de bbd100k a la de no_bloqueo

In [12]:
random.seed(42)

BASE = Path.cwd().parents[0] / "dataset" / "imagenes_dataset"
DESTINO = Path.cwd().parents[0] / "dataset" / "dataset_bloqueos"
EXT_VALIDAS = [".jpg", ".jpeg", ".png"]
splits = {"train": 0.7, "val": 0.15, "test": 0.15}

def recolectar_imagenes(carpeta_dataset, max_samples=None):
    imgs = []
    for split_name in ["train", "valid", "test"]:
        carpeta_imgs = BASE / carpeta_dataset / split_name / "images"
        if carpeta_imgs.exists():
            for f in carpeta_imgs.glob("*"):
                if f.suffix.lower() in EXT_VALIDAS:
                    imgs.append(f)
    if max_samples is not None and len(imgs) > max_samples:
        imgs = random.sample(imgs, max_samples)
    return imgs

def dividir_y_agregar(imgs, clase, prefijo_fuente):
    random.shuffle(imgs)
    n = len(imgs)
    n_train = int(n * splits["train"])
    n_val = int(n * splits["val"])

    reparto = {
        "train": imgs[:n_train],
        "val": imgs[n_train:n_train + n_val],
        "test": imgs[n_train + n_val:],
    }

    for split, lista in reparto.items():
        carpeta_out = DESTINO / split / clase
        carpeta_out.mkdir(parents=True, exist_ok=True)
        for i, img_path in enumerate(lista):
            nuevo_nombre = f"{prefijo_fuente}_{i}{img_path.suffix.lower()}"
            shutil.copy(img_path, carpeta_out / nuevo_nombre)

    print(f"[{prefijo_fuente}] total={n} -> train={len(reparto['train'])}, val={len(reparto['val'])}, test={len(reparto['test'])}")

fuentes = {
    "bloqueo": {
        "01_CONOS": 300,
        "03_LLANTAS": 300,
        "04_ACCIDENTES": 600,
    },
    "no_bloqueo": {
        "02_TRAFFIC_FLOW": None,   
        "05_BDD100K": None,       
    }
}

for clase, origenes in fuentes.items():
    for carpeta_origen, max_s in origenes.items():
        imgs = recolectar_imagenes(carpeta_origen, max_s)
        dividir_y_agregar(imgs, clase, prefijo_fuente=carpeta_origen.lower())


[01_conos] total=300 -> train=210, val=45, test=45
[03_llantas] total=300 -> train=210, val=45, test=45
[04_accidentes] total=600 -> train=420, val=90, test=90
[02_traffic_flow] total=726 -> train=508, val=108, test=110
[05_bdd100k] total=1484 -> train=1038, val=222, test=224


In [18]:
BASE = Path.cwd().parents[0] / "dataset" / "dataset_bloqueos"
carpetas = ["train", "val", "test"]
estado = ["bloqueo","no_bloqueo"]

for c in carpetas:
    for e in estado:
        total = 0
        p = BASE / c / e 
        if p.exists():
            n = len(list(p.glob("*")))
            total += n
        print(f"{c} - {e}: {total} imágenes")

train - bloqueo: 1900 imágenes
train - no_bloqueo: 1546 imágenes
val - bloqueo: 225 imágenes
val - no_bloqueo: 330 imágenes
test - bloqueo: 226 imágenes
test - no_bloqueo: 334 imágenes
